## 第一节
### membership constraints
* **方法与步骤**

    * 1.通过比较标准名单，提取不合法的类别，存入`inconsistent_categories`。
    * 2.查看目标列类别是否有不合法的类别（`isin()`），并输出布尔序列，存入`is_dirty`。
    * 3.用 is_dirty 序列作为索引，筛选出 study_data 中的异常行。
    * 4.通过'~'取反，取出正常数据
    * 5.`assert`验收清洗后的类别列是否是标准库的子集(.issubset())
    


In [ ]:
import pandas as pd

# 1. 亲手造出“脏数据” study_data
study_data = pd.DataFrame({
    'name': ['Ahmed', 'Josie', 'Tara', 'Humberto'],
    'birthday': ['1992-05-01', '1988-12-12', '1995-10-23', '1990-03-15'],
    'blood_type': ['A-', 'Z+', 'O+', 'AB+']  # 看，Z+ 在这里！
})

# 2. 建立“标准答案表” categories
categories = pd.DataFrame({
    'blood_type': ['O-', 'O+', 'A-', 'A+', 'B-', 'B+', 'AB-', 'AB+']
})

# 3. 开始你的实验：找出 Z+
inconsistent_categories = set(study_data['blood_type']).difference(set(categories['blood_type']))
print(f"找出的不合法血型: {inconsistent_categories}")

# 4.查看目标列中的类别是否有不合法的类别，输出布尔序列，存入is_dirty
# # 这是一个布尔掩码 (Boolean Mask)
is_dirty = study_data['blood_type'].isin(inconsistent_categories)
print(is_dirty)

# 5.提取有不合法类别的那一行数据，进行人工审计
dirty_data = study_data[is_dirty]
print("\n--- 异常数据行 ---")
print(dirty_data)

# 6.利用'~'取反，提取干净数据
clean_data = study_data[~is_dirty]
print("\n--- 清洗后的数据 ---")
print(clean_data)

# --- 验收环节 ---
# 确保洗完后的数据中，唯一值的集合是标准名单的子集
assert set(clean_data['blood_type']).issubset(set(categories['blood_type']))
print("\n✅ 数据清洗验收通过！")

找出的不合法血型: {'Z+'}
0    False
1     True
2    False
3    False
Name: blood_type, dtype: bool

--- 异常数据行 ---
    name    birthday blood_type
1  Josie  1988-12-12         Z+

--- 清洗后的数据 ---
       name    birthday blood_type
0     Ahmed  1992-05-01         A-
2      Tara  1995-10-23         O+
3  Humberto  1990-03-15        AB+

✅ 数据清洗验收通过！


## 第二节：分类变量转换 (Categorical Variables)
* **分箱 (Binning)：**

    * cut：按数值区间切（如：按年龄段 0-18, 18-60）。

    * qcut：按比例平分（如：把用户按活跃度平均分成 5 组）。

* **字符串清理：**

    | 方法 | 功能 | 解决问题 |
    | :--- | :---: | :---: |
    | .str.strip() | 去除首尾空格 | 解决 'Clean ' 和 'Clean' 被识别为两类的问题 |
    | .str.lower() | 统一转为小写 | 解决 'Clean', 'clean', 'CLEAN' 的大小写混乱 |
    | .str.replace() | 批量替换字符 | 将非标准描述（如 'Very-Dirty') 统一为标准项 ('Dirty') |

* **`.str` 的必要性：**
    * 在对 pandas 列（Series）进行 `strip(), lower(), replace()` 操作时，必须先写 `.str`。因为这些是字符串方法，`如果不加 .str`，pandas 会以为在对整个“列对象”操作，从而报错。

* **replace() 的强大之处：**
    * 如果有很多个要替换的项，可以用字典格式：`df.replace({'A': 'a', 'B': 'b'})`。

* **顺序很重要：**
    * 通常建议先 `strip()`（去空格），再 `lower()`（变小写），最后再 `replace()`。因为如果字符串前后有空格，replace 有可能因为匹配不上而失效

In [9]:
# cut与qcut的区别
import pandas as pd

# 1. 构造一组连续数据：20个人的评分
scores = [10, 15, 20, 25, 30, 45, 50, 55, 60, 70, 75, 80, 85, 90, 92, 95, 97, 98, 99, 100]
df = pd.DataFrame({'score': scores})

# --- 方法 A: pd.cut (定义固定的边界) ---
# 定义边界：0-60(不及格), 60-85(良), 85-100(优)
group_names = ['Fail', 'Good', 'Excellent']
df['cut_group'] = pd.cut(df['score'], bins=[0, 60, 85, 100], labels=group_names)
print(f"打印cut_group:")
print(df)
# --- 方法 B: pd.qcut (按比例平分，分成4份) ---
# 不需要定义边界，只需要说你想分成几份 (q=4)
df['qcut_group'] = pd.qcut(df['score'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(f"打印qcut_group:")
print(df)

print("--- cut 结果 (按刻度) ---")
print(df['cut_group'].value_counts()) # 你会发现 Fail 的人最多

print("\n--- qcut 结果 (按人头) ---")
print(df['qcut_group'].value_counts()) # 你会发现每个组都是 5 个人

打印cut_group:
    score  cut_group
0      10       Fail
1      15       Fail
2      20       Fail
3      25       Fail
4      30       Fail
5      45       Fail
6      50       Fail
7      55       Fail
8      60       Fail
9      70       Good
10     75       Good
11     80       Good
12     85       Good
13     90  Excellent
14     92  Excellent
15     95  Excellent
16     97  Excellent
17     98  Excellent
18     99  Excellent
19    100  Excellent
打印qcut_group:
    score  cut_group qcut_group
0      10       Fail         Q1
1      15       Fail         Q1
2      20       Fail         Q1
3      25       Fail         Q1
4      30       Fail         Q1
5      45       Fail         Q2
6      50       Fail         Q2
7      55       Fail         Q2
8      60       Fail         Q2
9      70       Good         Q2
10     75       Good         Q3
11     80       Good         Q3
12     85       Good         Q3
13     90  Excellent         Q3
14     92  Excellent         Q3
15     95  Excellent

In [10]:
import pandas as pd

# 1. 构造带有“马甲”的脏数据
# 注意：里面的 'Clean ' 有空格，'clean' 是小写，'DIRTY' 是大写
df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'cleanliness': ['Clean ', 'clean', 'Average', 'DIRTY', 'Very-Dirty']
})

print("--- 清洗前分类统计 ---")
print(df['cleanliness'].value_counts()) 
# 你会发现由于空格和大小写，'Clean' 被分成了两类

# 2. 开始标准清洗流程

# 第一步：去除首尾空格
df['cleanliness'] = df['cleanliness'].str.strip()

# 第二步：统一大小写（通常建议转小写，方便后续匹配）
df['cleanliness'] = df['cleanliness'].str.lower()

# 第三步：处理特殊的非标准描述
# 比如把 'very-dirty' 统一替换成 'dirty'
df['cleanliness'] = df['cleanliness'].replace({'very-dirty': 'dirty'})

print("\n--- 清洗后分类统计 ---")
print(df['cleanliness'].value_counts())

# --- 验证环节 ---
# 确保现在的分类只剩下我们预期的三种：clean, average, dirty
expected_categories = {'clean', 'average', 'dirty'}
assert set(df['cleanliness']).issubset(expected_categories)
print("\n✅ 字符串标准化完成！")

--- 清洗前分类统计 ---
cleanliness
Clean         1
clean         1
Average       1
DIRTY         1
Very-Dirty    1
Name: count, dtype: int64

--- 清洗后分类统计 ---
cleanliness
clean      2
dirty      2
average    1
Name: count, dtype: int64

✅ 字符串标准化完成！


## 第三节：清理非结构化文本数据（cleaning text data）
### 1. 核心方法与逻辑要点


* **第一招：链式替换 (Chained Replace)**
    * 利用 .str.replace() 像剥洋葱一样，一层层去掉不需要的符号。

    * **逻辑：** 把 + 变成 00，把 - 变成 ""（空字符串，相当于删除）。

* **第二招：长度过滤与强制缺失 (Length Filtering)**
    * 利用 .str.len() 识别出那些“一眼假”的数据。

    * **逻辑：** 如果电话号码长度小于 10 位，那肯定是错的，直接用 np.nan 把它们抹掉。

* **第三招：正则提取 (Regular Expressions - Regex) —— 本节大杀器**
    * 视频里提到了一个很强的正则模式：r'\D'。

    * **逻辑：** 不用一个一个去替换 +、-、(，而是直接告诉 Python：“除了数字（Digit）以外的所有东西，全部给我扔掉。”
### 2. 实际操作代码 (Practice Code)：

In [ ]:
import pandas as pd
import numpy as np

# 1. 构造“极度混乱”的电话数据
phones = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'David'],
    'phone_number': ['+86-138-1234-5678', '0086 139 8888 9999', '12345', '137-9999-1111']
})

# --- 方法 A: 传统替换法 (一步步来) ---
# 将 + 换成 00
phones['phone_cleaned'] = phones['phone_number'].str.replace('+', '00', regex=False)
# 去掉横杠
phones['phone_cleaned'] = phones['phone_cleaned'].str.replace('-', '', regex=False)

# --- 方法 B: 正则表达式法 (一次性解决) ---
# \D 代表 "Non-digit" (非数字)，把非数字全部换成空
phones['phone_regex'] = phones['phone_number'].str.replace(r'\D', '', regex=True)

# --- 长度清洗 ---
# 如果长度小于 10 位，视为无效数据，设为 NaN
# 注意：len 计算的是字符串长度
digits_len = phones['phone_regex'].str.len()
phones.loc[digits_len < 10, 'phone_regex'] = np.nan

print("--- 清洗后的电话簿 ---")
print(phones[['name', 'phone_regex']])

# --- 验收环节 (Assert) ---
# 1. 验证是否还有人带横杠
assert phones['phone_regex'].str.contains('-').any() == False
# 2. 验证剩下的号码是否都至少有 10 位 (排除掉 NaN)
assert phones['phone_regex'].dropna().str.len().min() >= 10

print("\n✅ 文本清洗验收通过！")

### 📝 正则表达式常用符号对比

| 符号 | 匹配内容 | 记忆技巧 | 常用场景 |
| :--- | :--- | :--- | :--- |
| `\d+` | 连续数字 | **d**igits (小写=本人) | 提取纯号码 |
| `\D+` | 连续**非**数字 | 大写=取反 (敌人) | 去除干扰字符 (+, -, 空格) |
| `\s+` | 连续空格 | **s**pace | 处理多余空格 |
| `\w+` | 字母数字下划线 | **w**ords | 提取用户名/ID |

### 🔍 文本检测与逻辑验证工具

| 方法 | 语法示例 | 作用 | 返回类型 |
| :--- | :--- | :--- | :--- |
| **`.contains()`** | `df['col'].str.contains('关键词')` | 模糊搜索，查找是否包含特定文本或正则模式 | 布尔序列 (True/False) |
| **`.any()`** | `series.any()` | 检查序列中是否有**至少一个** `True` | 单个布尔值 |
| **`.all()`** | `series.all()` | 检查序列中是否**全部**都是 `True` | 单个布尔值 |

* **💡 搭配使用的“黄金公式”**
  
在写 assert 验收代码时，会经常用到它们的组合：

* **检查是否所有行都洗干净了（不含特定符号）：**
  
`assert (df['col'].str.contains(r'[+|-]').any() == False)`

解释：只要有一个带符号的（any），断言就会报错。

* **检查是否所有行都满足长度要求：**
`assert (df['col'].str.len() >= 10).all()`

解释：必须全部（all）大于等于10位，断言才通过。

# 🏁 第二章总结：数据清洗实战流水线

| 维度 | 清洗动作 (Action) | 核心工具 (Tools) | 验收手段 (Validation) |
| :--- | :--- | :--- | :--- |
| **类别一致性** | 剔除不在标准名单里的离群值 | `set().difference()`, `.isin()`, `~` | `assert ... .issubset()` |
| **数值分箱** | 将连续数字切分为离散标签 | `pd.cut` (等距), `pd.qcut` (等频) | `.value_counts()` |
| **文本标准化** | 统一大小写、去空格、马甲替换 | `.str.strip()`, `.str.lower()`, `.replace()` | `.unique()` |
| **非结构化清理** | 暴力提取数字、处理异常长度 | `r'\D'`, `.str.len()`, `np.nan` | `assert ... .str.contains().any()` |

### 🛠️ 核心代码快查 (Cheat Sheet)

#### 1. 成员资格过滤 (Membership Constraints)
```python
# 找出不在 categories 里的脏数据并剔除
inconsistent = set(df['col']).difference(set(categories['col']))
mask = df['col'].isin(inconsistent)
clean_df = df[~mask]
```

#### 2. 字符串净化 (String Manipulation)
```python
# 黄金三部曲：去空格 -> 小写化 -> 异常替换
df['col'] = df['col'].str.strip().str.lower().replace({'old': 'new'})
```

#### 3. 正则表达式清洗 (Regex Cleaning)
```python
# 仅保留数字，并处理长度不足的无效值
df['phone'] = df['phone'].str.replace(r'\D', '', regex=True)
df.loc[df['phone'].str.len() < 10, 'phone'] = np.nan
```